In [ ]:
from notebook_utils import load_hydra_config, to_pil
from when2plan.envs.doorkey.env import DoorKeyEnv
from omegaconf import OmegaConf
import numpy as np
from IPython.display import display, clear_output
import numpy as np
import torch as th
import time
import copy
from when2plan.utils import collect_render_data_with_planner, get_model_free_metrics, plot_model_free_metrics
from when2plan.models.pretrain_model_free import pretrain_model_free_sub_env
from argparse import Namespace
from when2plan.envs.meta_env import MetaEnv

In [ ]:
in_distribution_cfg = load_hydra_config("../experiments/configs", "config", ["env=doorkey", 'experiment=doorkey_pretrain'])
out_of_distribution_cfg = load_hydra_config("../experiments/configs", "config", ["env=doorkey", 'experiment=doorkey_pretrain'])

num_tasks = 100
in_distribution_cfg.experiment.num_cached_tasks = num_tasks * 2
out_of_distribution_cfg.experiment.num_cached_tasks = num_tasks * 2

# Set in distribution vs out of distribution parameters
doorkey_id = DoorKeyEnv(in_distribution_cfg)
doorkey_ood = DoorKeyEnv(out_of_distribution_cfg)

doorkey_id.set_cache_task_range(0, in_distribution_cfg.experiment.num_pretrained_tasks)
doorkey_ood.set_cache_task_range(0, 20)
 

_, _ = doorkey_id.reset()
_, _ = doorkey_ood.reset()

In [ ]:
# Pretrain model free agent on in-distribution environment
args = Namespace(
    sub_env="doorkey",
    model_free_action_space="discrete",
    model_free_observation_space="tensor",
    num_actors=4,  
    steps=10000,
    batch_size=64,
    epochs=30,
    base_seed=42,
    plot=True,
    save_actor=True,
)
pretrained_networks, env = pretrain_model_free_sub_env(args, doorkey_id)

In [ ]:
# Create meta environments for in-distribution and out-of-distribution
id_meta_env = MetaEnv(in_distribution_cfg, doorkey_id, pretrained_networks)
ood_meta_env = MetaEnv(out_of_distribution_cfg, doorkey_ood, pretrained_networks)

# Get type of meta step from config
model_free_action = 0

# Get metrics
metrics = get_model_free_metrics(id_meta_env, ood_meta_env, n_steps=5000, model_free_action=model_free_action)
plot_model_free_metrics(metrics, save_path='doorkey_model_free_metrics.png')